# NFL Draft Prediction - Final Model Pipeline
This notebook implements the exact pipeline that generated the 0.84653 submission.
It combines a **PyTorch Neural Network** (with Entity Embeddings) and a **CatBoost** model.
It then uses highly confident test-set predictions (Pseudo-Labeling) to augment the training data and explicitly learn the test distribution without leaking data into the early stopping validations.

Run this notebook to generate `submission.csv`.

In [ ]:
# NFL Draft Prediction - Final Model Pipeline
# This notebook implements the V6 Pseudo-Labeling Hybrid Model
# It combines a PyTorch Neural Network with CatBoost and uses Semi-Supervised Learning.

import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier, Pool

warnings.filterwarnings("ignore")
device = torch.device("cpu")

# 1. LOAD DATA
train = pd.read_csv("Data/train.csv")
test = pd.read_csv("Data/test.csv")
y = train["Drafted"].reset_index(drop=True)
test_ids = test["Id"].reset_index(drop=True)

CAT_COLS = ["School", "Player_Type", "Position_Type", "Position"]
PHYSICAL = [
    "Height", "Weight", "Sprint_40yd", "Vertical_Jump",
    "Bench_Press_Reps", "Broad_Jump", "Agility_3cone", "Shuttle", "Age"
]

CB_PARAMS = {
    'iterations': 1183, 'learning_rate': 0.01049, 'depth': 4,
    'l2_leaf_reg': 4.559, 'min_data_in_leaf': 7, 'random_seed': 42,
    'eval_metric': 'Logloss', 'verbose': 0
}

# 2. FEATURE ENGINEERING
def build_features(train_df, test_df):
    df = pd.concat([train_df.drop(columns=["Drafted"]), test_df], axis=0).reset_index(drop=True)
    for col in PHYSICAL:
        if df[col].isnull().sum() > 0:
            df[f"{col}_Is_Missing"] = df[col].isnull().astype(int)
    for col in PHYSICAL:
        df[col] = df[col].fillna(df.groupby("Position")[col].transform("median")).fillna(df[col].median())
    
    df["BMI"] = df["Weight"] / (df["Height"] ** 2)
    df["Explosiveness_Index"] = df["Vertical_Jump"] * df["Broad_Jump"]
    df["Power_Score"] = df["Bench_Press_Reps"] * df["Weight"]
    df["Speed_Score"] = (df["Weight"] * 200) / (df["Sprint_40yd"] ** 4)
    
    for col in PHYSICAL:
        df[f"{col}_Pos_Mean"] = df.groupby("Position")[col].transform("mean")
        df[f"{col}_Pos_Diff"] = df[col] - df[f"{col}_Pos_Mean"]
        df.drop(columns=[f"{col}_Pos_Mean"], inplace=True)

    df_cb = df.copy()
    for col in CAT_COLS: df_cb[col] = df_cb[col].fillna("Unknown").astype(str)
    X_train_cb = df_cb.iloc[:len(train_df)].reset_index(drop=True)
    X_test_cb = df_cb.iloc[len(train_df):].reset_index(drop=True)

    df_nn = df.copy()
    cat_dims = {}
    for col in CAT_COLS:
        df_nn[col] = df_nn[col].fillna("Unknown").astype(str)
        df_nn[col], uniques = pd.factorize(df_nn[col])
        cat_dims[col] = len(uniques)
    
    cont_cols = [c for c in df_nn.columns if c not in CAT_COLS + ["Id"]]
    df_nn[cont_cols] = StandardScaler().fit_transform(df_nn[cont_cols])
    df_nn[cont_cols] = df_nn[cont_cols].fillna(0)
    
    X_train_nn = df_nn.iloc[:len(train_df)].reset_index(drop=True)
    X_test_nn = df_nn.iloc[len(train_df):].reset_index(drop=True)
    
    return X_train_cb, X_test_cb, X_train_nn, X_test_nn, cont_cols, cat_dims

X_train_cb, X_test_cb, X_train_nn, X_test_nn, cont_cols, cat_dims = build_features(train, test)

# 3. PYTORCH NEURAL NETWORK DEFINITION
class TabularDataset(Dataset):
    def __init__(self, df, cont_cols, cat_cols, y=None):
        self.cont = torch.tensor(df[cont_cols].values, dtype=torch.float32)
        self.cat = torch.tensor(df[cat_cols].values, dtype=torch.long)
        self.y = torch.tensor(y.values, dtype=torch.float32) if y is not None else None
    def __len__(self): return len(self.cont)
    def __getitem__(self, idx):
        if self.y is not None: return self.cont[idx], self.cat[idx], self.y[idx]
        return self.cont[idx], self.cat[idx]

class TabularNN(nn.Module):
    def __init__(self, cat_dims, num_cont, hidden_dims=[256, 128]):
        super().__init__()
        self.embeds = nn.ModuleList([nn.Embedding(num_classes, min(50, (num_classes + 1) // 2)) for col, num_classes in cat_dims.items()])
        in_dim = sum(e.embedding_dim for e in self.embeds) + num_cont
        layers = []
        for out_dim in hidden_dims:
            layers.extend([nn.Linear(in_dim, out_dim), nn.BatchNorm1d(out_dim), nn.GELU(), nn.Dropout(0.3)])
            in_dim = out_dim
        layers.append(nn.Linear(in_dim, 1))
        self.mlp = nn.Sequential(*layers)
    def forward(self, x_cont, x_cat):
        emb = torch.cat([e(x_cat[:, i]) for i, e in enumerate(self.embeds)], dim=1)
        return self.mlp(torch.cat([x_cont, emb], dim=1)).squeeze(1)

# 4. TRAINING FUNCTIONS
def train_nn(X, X_test, y, pseudo_X=None, pseudo_y=None):
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    test_pred = np.zeros(len(X_test))
    test_dl = DataLoader(TabularDataset(X_test, cont_cols, CAT_COLS), batch_size=256, shuffle=False)
    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y)):
        X_tr, y_tr = X.iloc[tr_idx].copy(), y.iloc[tr_idx].copy()
        X_va, y_va = X.iloc[va_idx].copy(), y.iloc[va_idx].copy()
        if pseudo_X is not None and len(pseudo_X) > 0:
            X_tr = pd.concat([X_tr, pseudo_X], axis=0).reset_index(drop=True)
            y_tr = pd.concat([y_tr, pseudo_y], axis=0).reset_index(drop=True)
        tr_dl = DataLoader(TabularDataset(X_tr, cont_cols, CAT_COLS, y_tr), batch_size=128, shuffle=True)
        va_dl = DataLoader(TabularDataset(X_va, cont_cols, CAT_COLS, y_va), batch_size=256, shuffle=False)
        model = TabularNN(cat_dims, len(cont_cols)).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
        best_va_auc, patience = 0, 0
        best_test_preds = []
        for epoch in range(50):
            model.train()
            for x_cont, x_cat, targets in tr_dl:
                optimizer.zero_grad()
                loss = nn.BCEWithLogitsLoss()(model(x_cont.to(device), x_cat.to(device)), targets.to(device))
                loss.backward()
                optimizer.step()
            model.eval()
            va_preds, va_targets = [], []
            with torch.no_grad():
                for x_cont, x_cat, targets in va_dl:
                    va_preds.extend(torch.sigmoid(model(x_cont.to(device), x_cat.to(device))).cpu().numpy())
                    va_targets.extend(targets.numpy())
            auc = roc_auc_score(va_targets, va_preds)
            if auc > best_va_auc:
                best_va_auc, patience = auc, 0
                best_test_preds = []
                with torch.no_grad():
                    for x_cont, x_cat in test_dl:
                        best_test_preds.extend(torch.sigmoid(model(x_cont.to(device), x_cat.to(device))).cpu().numpy())
            else:
                patience += 1
            if patience >= 5: break
        test_pred += np.array(best_test_preds) / 10
    return test_pred

def train_cb(X, X_test, y, pseudo_X=None, pseudo_y=None):
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    test_pred = np.zeros(len(X_test))
    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), 1):
        X_tr, y_tr = X.iloc[tr_idx].copy(), y.iloc[tr_idx].copy()
        X_va, y_va = X.iloc[va_idx].copy(), y.iloc[va_idx].copy()
        if pseudo_X is not None and len(pseudo_X) > 0:
            X_tr = pd.concat([X_tr, pseudo_X], axis=0).reset_index(drop=True)
            y_tr = pd.concat([y_tr, pseudo_y], axis=0).reset_index(drop=True)
        tp = Pool(X_tr.drop(columns=["Id"]), y_tr, cat_features=CAT_COLS)
        vp = Pool(X_va.drop(columns=["Id"]), y_va, cat_features=CAT_COLS)
        p = CB_PARAMS.copy()
        p["random_seed"] = 42 + fold
        model = CatBoostClassifier(**p)
        model.fit(tp, eval_set=vp, early_stopping_rounds=50)
        test_pred += model.predict_proba(Pool(X_test.drop(columns=["Id"]), cat_features=CAT_COLS))[:, 1] / 10
    return test_pred

# 5. EXECUTION & PSEUDO-LABELING
print("Training Base Models...")
base_test_cb = train_cb(X_train_cb, X_test_cb, y)
base_test_nn = train_nn(X_train_nn, X_test_nn, y)
base_test_blend = (pd.Series(base_test_cb).rank(pct=True) + pd.Series(base_test_nn).rank(pct=True)) / 2

# Bottleneck Pseudo Labeling
pos_idx = np.flatnonzero(base_test_blend >= 0.90)
neg_idx = np.flatnonzero(base_test_blend <= 0.10)
capped_pos = min(len(pos_idx), int(round(len(neg_idx) * (y.mean() / (1 - y.mean())))))
selected_idx = np.concatenate([pos_idx[:capped_pos], neg_idx])
pseudo_labels = np.concatenate([np.ones(capped_pos), np.zeros(len(neg_idx))])
pseudo_y = pd.Series(pseudo_labels, index=selected_idx, dtype=int)

print("Training Pseudo Models...")
ps_test_cb = train_cb(X_train_cb, X_test_cb, y, X_test_cb.iloc[selected_idx], pseudo_y)
ps_test_nn = train_nn(X_train_nn, X_test_nn, y, X_test_nn.iloc[selected_idx], pseudo_y)
final_blend = (pd.Series(ps_test_cb).rank(pct=True) + pd.Series(ps_test_nn).rank(pct=True)) / 2

pd.DataFrame({"Id": test_ids, "Drafted": final_blend}).to_csv("submission.csv", index=False)
print("Submission saved to submission.csv!")
